# Hyperparameter Optimization

In [ ]:
import seaborn as sns
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
import optuna.visualization as vis

from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical

import sys 
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig

%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Load / Reload Selection Utility Functions

In [ ]:
from utils2.optimization import *

----

## Read Config File

In [ ]:
config_path = Path(r'experiments\binary')
config_file = config_path / "optimization_config_dev.yml"
#config_file = config_path / "optimization_config_final.yml"
config_dict = ymlconfig.load_config(config_file)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict

## Data Loading

In [ ]:
D = DPN_data(config.data.dataset_path)
D.load(classification=config.experiment.classification_type)

dfdpn = D.df
data_cols = dfdpn.drop(D.non_data_cols, axis=1, errors="ignore").columns
X = dfdpn[data_cols]
y = dfdpn['Confirmed_Binary_DPN']
X.shape, y.shape

## Bayesian Optimization of Top 2 Models

## CatBoost Bayesian Optimization (Top 1 Model)

----

### Working

In [ ]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.metrics import roc_curve, confusion_matrix, roc_auc_score
from skopt import BayesSearchCV


def nested_cv_youden(
    X,
    y,
    model,
    param_space,
    n_splits_outer=3,
    n_repeats_outer=3,
    n_splits_inner=3,
    n_iter=30,
    random_state=42,
    n_jobs=-1,
):
    """
    Nested cross-validation with:
    - Outer: RepeatedStratifiedKFold
    - Inner: BayesSearchCV optimizing ROC-AUC
    - Threshold selected on inner validation data using Youden index,
      then applied to the outer test fold (avoids optimistic bias)

    Returns:
        dict with mean/std summary metrics and per-fold results
    """

    outer_cv = RepeatedStratifiedKFold(
        n_splits=n_splits_outer,
        n_repeats=n_repeats_outer,
        random_state=random_state,
    )

    fold_results = []

    for fold_idx, (outer_train_idx, outer_test_idx) in enumerate(
        outer_cv.split(X, y)
    ):
        X_outer_train, X_outer_test = X[outer_train_idx], X[outer_test_idx]
        y_outer_train, y_outer_test = y[outer_train_idx], y[outer_test_idx]

        # ── Inner CV: hyperparameter search ──────────────────────────────────
        inner_cv = StratifiedKFold(
            n_splits=n_splits_inner,
            shuffle=True,
            random_state=random_state,
        )

        opt = BayesSearchCV(
            estimator=model,
            search_spaces=param_space,
            scoring="roc_auc",
            cv=inner_cv,
            n_iter=n_iter,
            n_jobs=n_jobs,
            random_state=random_state,
            refit=True,  # refit on full outer_train after search
        )
        opt.fit(X_outer_train, y_outer_train)
        best_model = opt.best_estimator_

        # ── Threshold selection on inner-CV OOF predictions ──────────────────
        # Collect out-of-fold probabilities across the inner splits so the
        # threshold is never chosen on data the model was trained on.
        oof_proba = np.zeros(len(y_outer_train))
        for inner_train_idx, inner_val_idx in inner_cv.split(
            X_outer_train, y_outer_train
        ):
            fold_model = opt.best_estimator_.__class__(
                **opt.best_params_
            )
            fold_model.fit(
                X_outer_train[inner_train_idx],
                y_outer_train[inner_train_idx],
            )
            oof_proba[inner_val_idx] = fold_model.predict_proba(
                X_outer_train[inner_val_idx]
            )[:, 1]

        fpr, tpr, threshold_candidates = roc_curve(y_outer_train, oof_proba)
        youden_index = tpr - fpr
        best_threshold = float(threshold_candidates[np.argmax(youden_index)])

        # ── Evaluation on outer test fold ─────────────────────────────────────
        y_test_proba = best_model.predict_proba(X_outer_test)[:, 1]
        y_test_pred = (y_test_proba >= best_threshold).astype(int)

        # Guard against degenerate folds (single class in test set)
        cm = confusion_matrix(y_outer_test, y_test_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        youden_test = (
            sensitivity + specificity - 1
            if not (np.isnan(sensitivity) or np.isnan(specificity))
            else np.nan
        )
        roc_auc = (
            roc_auc_score(y_outer_test, y_test_proba)
            if len(np.unique(y_outer_test)) > 1
            else np.nan
        )

        fold_results.append(
            {
                "fold": fold_idx,
                "roc_auc": roc_auc,
                "youden": youden_test,
                "sensitivity": sensitivity,
                "specificity": specificity,
                "threshold": best_threshold,
                "best_params": dict(opt.best_params_),
            }
        )

    # ── Aggregate ─────────────────────────────────────────────────────────────
    def _nanstats(key):
        vals = [f[key] for f in fold_results]
        return float(np.nanmean(vals)), float(np.nanstd(vals))

    roc_mean, roc_std = _nanstats("roc_auc")
    you_mean, you_std = _nanstats("youden")
    sen_mean, _ = _nanstats("sensitivity")
    spe_mean, _ = _nanstats("specificity")
    thr_mean, thr_std = _nanstats("threshold")

    return {
        # ── Summary stats ──
        "roc_auc_mean": roc_mean,
        "roc_auc_std": roc_std,
        "youden_mean": you_mean,
        "youden_std": you_std,
        "sensitivity_mean": sen_mean,
        "specificity_mean": spe_mean,
        "threshold_mean": thr_mean,
        "threshold_std": thr_std,
        # ── Per-fold detail ──
        "folds": fold_results,
    }

In [ ]:
model = CatBoostClassifier(
    verbose=0,  # Show progress every 100 iterations
    loss_function="Logloss",
    eval_metric="AUC",
    random_state=42,
    thread_count=-1  # Use all CPU cores
)

param_space = {
    'iterations': [10], #[100, 200, 500],
    'depth': [4, 6], # [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05], #[0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3], # [1, 3, 5, 7, 9]
}

results = nested_cv_youden(
    X=X.values,
    y=y.values,
    model=model,
    param_space=param_space,
    n_splits_outer=5,
    n_repeats_outer=5,
    n_splits_inner=3,
    n_iter=30
)

print(f"Results: {results}")

0:	learn: 0.6740436	total: 4.32ms	remaining: 38.9ms
1:	learn: 0.6533032	total: 7.43ms	remaining: 29.7ms
2:	learn: 0.6318896	total: 11.2ms	remaining: 26.2ms
3:	learn: 0.6106520	total: 15.4ms	remaining: 23.2ms
4:	learn: 0.5930317	total: 19.3ms	remaining: 19.3ms
5:	learn: 0.5770169	total: 22.4ms	remaining: 14.9ms
6:	learn: 0.5605568	total: 26.8ms	remaining: 11.5ms
7:	learn: 0.5387206	total: 30.1ms	remaining: 7.51ms
8:	learn: 0.5237569	total: 33.8ms	remaining: 3.75ms
9:	learn: 0.5084875	total: 36.8ms	remaining: 0us
0:	learn: 0.6633255	total: 5.88ms	remaining: 52.9ms
1:	learn: 0.6422623	total: 9.44ms	remaining: 37.8ms
2:	learn: 0.6160746	total: 13ms	remaining: 30.4ms
3:	learn: 0.5910449	total: 16ms	remaining: 23.9ms
4:	learn: 0.5687358	total: 19.4ms	remaining: 19.4ms
5:	learn: 0.5506817	total: 23.5ms	remaining: 15.7ms
6:	learn: 0.5268314	total: 28.2ms	remaining: 12.1ms
7:	learn: 0.5149628	total: 31.2ms	remaining: 7.81ms
8:	learn: 0.4999916	total: 38.3ms	remaining: 4.26ms
9:	learn: 0.4835213

In [ ]:
results

{'roc_auc_mean': 0.9550641025641025,
 'roc_auc_std': 0.03577267697763135,
 'youden_mean': 0.8325641025641025,
 'youden_std': 0.0983813776255997,
 'sensitivity_mean': 0.9292307692307692,
 'specificity_mean': 0.9033333333333333,
 'threshold_mean': 0.5265682559445407,
 'threshold_std': 0.036808248147104226,
 'folds': [{'fold': 0,
   'roc_auc': 0.9615384615384616,
   'youden': 0.8461538461538463,
   'sensitivity': 0.8461538461538461,
   'specificity': 1.0,
   'threshold': 0.5423759056174232,
   'best_params': {'depth': 6,
    'iterations': 10,
    'l2_leaf_reg': 2,
    'learning_rate': 0.021790048159546157}},
  {'fold': 1,
   'roc_auc': 0.9775641025641025,
   'youden': 0.8397435897435899,
   'sensitivity': 0.9230769230769231,
   'specificity': 0.9166666666666666,
   'threshold': 0.5436899469122082,
   'best_params': {'depth': 5,
    'iterations': 10,
    'l2_leaf_reg': 1,
    'learning_rate': 0.027333312074809005}},
  {'fold': 2,
   'roc_auc': 0.9743589743589743,
   'youden': 0.83333333333

In [ ]:
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV
import numpy as np
from sklearn.metrics import roc_curve

def train_final_model(X, y, model, param_space, n_splits_inner=3, n_iter=30, random_state=42, n_jobs=-1):
    """
    Train the final deployable model on ALL data:
    1. BayesSearchCV to find best hyperparameters (inner CV on full dataset)
    2. Refit on full dataset with best params
    3. Threshold via Youden index on OOF predictions
    """

    inner_cv = StratifiedKFold(n_splits=n_splits_inner, shuffle=True, random_state=random_state)

    # Step 1: Hyperparameter search on full data
    opt = BayesSearchCV(
        estimator=model,
        search_spaces=param_space,
        scoring="roc_auc",
        cv=inner_cv,
        n_iter=n_iter,
        n_jobs=n_jobs,
        random_state=random_state,
        refit=True,  # fits final model on all data with best params
    )
    opt.fit(X, y)
    final_model = opt.best_estimator_

    # Step 2: Youden threshold via OOF probabilities on full dataset
    oof_proba = np.zeros(len(y))
    for inner_train_idx, inner_val_idx in inner_cv.split(X, y):
        fold_model = opt.best_estimator_.__class__(**opt.best_params_)
        fold_model.fit(X[inner_train_idx], y[inner_train_idx])
        oof_proba[inner_val_idx] = fold_model.predict_proba(X[inner_val_idx])[:, 1]

    fpr, tpr, thresholds = roc_curve(y, oof_proba)
    best_threshold = float(thresholds[np.argmax(tpr - fpr)])

    return final_model, best_threshold, opt.best_params_


# --- Run nested CV first to get honest performance estimates ---
results = nested_cv_youden(
    X=X.values,
    y=y.values,
    model=model,
    param_space=param_space,
    n_splits_outer=5,
    n_repeats_outer=5,
    n_splits_inner=3,
    n_iter=30
)

print(f"Estimated AUC: {results['roc_auc_mean']:.3f} ± {results['roc_auc_std']:.3f}")

# --- Then train final model on ALL data ---
final_model, final_threshold, best_params = train_final_model(
    X=X.values, y=y.values, model=model, param_space=param_space
)

# --- Deploy ---
def predict(X_new, model, threshold):
    proba = model.predict_proba(X_new)[:, 1]
    return (proba >= threshold).astype(int), proba

0:	learn: 0.6740436	total: 6.68ms	remaining: 60.1ms
1:	learn: 0.6533032	total: 12.2ms	remaining: 49ms
2:	learn: 0.6318896	total: 16.2ms	remaining: 37.9ms
3:	learn: 0.6106520	total: 20.2ms	remaining: 30.3ms
4:	learn: 0.5930317	total: 25.5ms	remaining: 25.5ms
5:	learn: 0.5770169	total: 29.7ms	remaining: 19.8ms
6:	learn: 0.5605568	total: 34ms	remaining: 14.6ms
7:	learn: 0.5387206	total: 38.4ms	remaining: 9.59ms
8:	learn: 0.5237569	total: 43.3ms	remaining: 4.81ms
9:	learn: 0.5084875	total: 47ms	remaining: 0us
0:	learn: 0.6633255	total: 5.07ms	remaining: 45.6ms
1:	learn: 0.6422623	total: 9.56ms	remaining: 38.3ms
2:	learn: 0.6160746	total: 14.4ms	remaining: 33.7ms
3:	learn: 0.5910449	total: 18.5ms	remaining: 27.7ms
4:	learn: 0.5687358	total: 22.3ms	remaining: 22.3ms
5:	learn: 0.5506817	total: 26ms	remaining: 17.4ms
6:	learn: 0.5268314	total: 31.8ms	remaining: 13.6ms
7:	learn: 0.5149628	total: 35.6ms	remaining: 8.91ms
8:	learn: 0.4999916	total: 39.4ms	remaining: 4.37ms
9:	learn: 0.4835213	tot

In [ ]:
final_model, final_threshold

(<catboost.core.CatBoostClassifier at 0x1d2e83c29c0>, 0.5251054350253742)

In [ ]:
test_size = 20
Xnew = X.iloc[:test_size].values
ypred, ypredproba = predict(Xnew, final_model, final_threshold)
ynew=y.iloc[:test_size].values

cm = confusion_matrix(ynew, ypred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
youden_test = (
    sensitivity + specificity - 1
    if not (np.isnan(sensitivity) or np.isnan(specificity))
    else np.nan
)
roc_auc = (
    roc_auc_score(ypred, ypredproba)
    if len(np.unique(ynew)) > 1
    else np.nan
)

print(youden_test, roc_auc)

0.9333333333333333 1.0


In [ ]:
import numpy as np

def mean_confidence_interval(results, metric, confidence=0.95):
    scores = [fold[metric] for fold in results['folds']]
    
    scores = np.array(scores)
    n = len(scores)
    mean = np.mean(scores)
    std = np.std(scores, ddof=1)
    stderr = std / np.sqrt(n)

    z = 1.96  # 95% normal approx
    margin = z * stderr

    return {
        "mean": mean,
        "std": std,
        "ci_lower": mean - margin,
        "ci_upper": mean + margin,
        "n_folds": n
    }

In [ ]:
youden_ci = mean_confidence_interval(results, "youden")
roc_ci = mean_confidence_interval(results, "roc_auc")

print("Youden 95% CI:", youden_ci)
print("ROC-AUC 95% CI:", roc_ci)

Youden 95% CI: {'mean': 0.8325641025641025, 'std': 0.10041007307282705, 'ci_lower': 0.7932033539195543, 'ci_upper': 0.8719248512086507, 'n_folds': 25}
ROC-AUC 95% CI: {'mean': 0.9550641025641025, 'std': 0.03651033555358497, 'ci_lower': 0.9407520510270971, 'ci_upper': 0.9693761541011078, 'n_folds': 25}


In [ ]:
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV

# Inner CV for full-data training
inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

final_search = BayesSearchCV(
    estimator=model,
    search_spaces=param_space,
    scoring="roc_auc",
    cv=inner_cv,
    n_iter=30,
    n_jobs=-1,
    random_state=42
)

final_search.fit(X, y)

final_model = final_search.best_estimator_

In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

y_proba = final_model.predict_proba(X)[:, 1]

fpr, tpr, thresholds = roc_curve(y, y_proba)
youden = tpr - fpr

final_threshold = thresholds[np.argmax(youden)]

In [ ]:
def predict_with_threshold(model, threshold, X_new):
    proba = model.predict_proba(X_new)[:, 1]
    return (proba >= threshold).astype(int)

In [ ]:
y_pred = predict_with_threshold(final_model, final_threshold, X_new)


In [ ]:
final_threshold

0.5085610025503904

 -----

## Model Evaluation Preparation

#### Train Test Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size, random_state=0, stratify=y)

#### Evaluate the Model

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall (Sensitivity)": recall_score(y_test, y_pred),
        "Specificity": specificity_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
        "Youden Index": youden_index_score(y_test, y_pred)
    }
    return pd.Series(metrics)

## Model Evaluation

In [ ]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_val)
y_proba = best_model.predict_proba(X_val)[:, 1]
results = evaluate_model(best_model, X_val, y_val)
print(results)

#### Saving the Optimized Model

In [ ]:
def save_optimized_results(name, best_params, best_score, optimized_model, optimized_model_metrics):
    model_results = { 
        "name" : name,
        "best_params": best_params, 
        "best_score": best_score, 
        "optimized_model": optimized_model, 
        "optimized_model_metrics": optimized_model_metrics
    }
    joblib.dump(model_results, rf"outputs\all_features\{name}.pkl")

In [ ]:
save_optimized_results("random_forest", opt.best_params_, opt.best_score_, best_model, results)